### Converting .flv files to .mp4s and applying OpenFace

In [32]:
import os               # VSCode Navigator
import subprocess       # Terminal Access
import pandas as pd

video_dir = '/Users/songye/Desktop/Dev/REU/CREMA-D/VideoFlash'
mp4_dir = '/Users/songye/Desktop/Dev/REU/CREMA-D/mp4'
output_dir = '/Users/songye/Desktop/Dev/REU/CREMA-D/output'


os.makedirs(mp4_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)


# A list of all the files
flv_video_files = [video for video in os.listdir(video_dir) if video.endswith('.flv')][:100]

for video in flv_video_files:

    # This converts the .flv file into mp4 files for OpenFace
    mp4_video_name = video.replace(".flv", ".mp4")
    subprocess.run([
        'ffmpeg',
        '-i',
        os.path.join(video_dir, video),
        '-y',
        '-loglevel',
        'error',
        os.path.join(mp4_dir, mp4_video_name)])
    

    # subprocess.run([
    # "docker", "run", "--rm",
    # "--platform", "linux/amd64",  # explicitly force amd64 emulation
    # "-v", "/Users/songye/Desktop/Dev/REU/CREMA-D:/data",
    # "algebr/openface:latest",
    # "/home/openface-build/build/bin/FeatureExtraction",
    # "-f", "/data/mp4/" + mp4_video_name,
    # "-out_dir", "/data/output/"])

    # This did not work, so have to run Docker manually via Terminal
    # docker run -it -v /Users/songye/Desktop/Dev/REU/CREMA-D:/data algebr/openface:latest
    # for f in /data/mp4/*.mp4; do /home/openface-build/build/bin/FeatureExtraction -f "$f" -out_dir /data/output/; done

    

### Constructing the Feature Matrix

In [31]:
# Now we need a feature matrix
# Each row = one entire video, columns = aggregated AU statistics + emotion label (Each AU's mean and std)
list_of_csvs = [csv for csv in os.listdir(output_dir) if csv.endswith(".csv")]
feature_matrix = pd.DataFrame()

count = 0
cols_order = []

for csv in list_of_csvs:
    df = pd.read_csv(os.path.join(output_dir, csv))
    df.columns = df.columns.str.strip()

    list_means = []
    list_std = []

    for col in df.columns:
        if col.startswith('AU') and (col.endswith('_r') or col.endswith('_c')):
            list_means.append(df[col].mean())
            list_std.append(df[col].std())
            if count == 0:
                cols_order.append(col)

    total_list = list_means.copy()
    total_list.extend(list_std)

    # To get the emotion from the file name
    emotion = csv.split('_')[2]
    total_list.append(emotion)

    feature_matrix = pd.concat([feature_matrix, pd.DataFrame([total_list])], ignore_index=True)
    count += 1

col_names = ([col + '_mean' for col in cols_order] + 
             [col + '_std' for col in cols_order] + 
             ['Emotion'])

feature_matrix.columns = col_names

#print(feature_matrix.head(5))
feature_matrix.info

<bound method DataFrame.info of      AU01_r_mean  AU02_r_mean  AU04_r_mean  AU05_r_mean  AU06_r_mean  \
0       0.056716     0.020597     1.141493     0.024179     0.333881   
1       0.125700     0.080600     1.429700     0.024300     0.337800   
2       0.092821     0.027821     0.046795     0.021538     0.027436   
3       0.048636     0.022121     0.046364     0.013485     0.000000   
4       0.102321     0.027143     0.000000     0.010357     0.393393   
..           ...          ...          ...          ...          ...   
96      0.113239     0.031831     0.000000     0.020563     0.000000   
97      0.240182     0.175455     0.101091     0.016000     0.233636   
98      0.341098     0.200976     0.000000     0.044146     0.032683   
99      0.125352     0.036620     0.155634     0.024648     0.000000   
100     0.105057     0.074943     0.401954     0.034023     0.049080   

     AU07_r_mean  AU09_r_mean  AU10_r_mean  AU12_r_mean  AU14_r_mean  ...  \
0       0.224925     0.023

### Model Training

In [30]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

X = feature_matrix.drop(columns = ["Emotion"])
y = feature_matrix["Emotion"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

classifier = SVC(kernel = 'rbf', C = 0.8, gamma = 'scale')
classifier.fit(X_train_scaled, y_train)
y_pred = classifier.predict(X_test_scaled)

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.41935483870967744
              precision    recall  f1-score   support

         ANG       1.00      0.50      0.67         4
         DIS       0.00      0.00      0.00         4
         FEA       0.13      1.00      0.24         2
         HAP       0.82      0.75      0.78        12
         NEU       0.00      0.00      0.00         0
         SAD       0.00      0.00      0.00         9

    accuracy                           0.42        31
   macro avg       0.33      0.38      0.28        31
weighted avg       0.45      0.42      0.40        31



/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  